In [0]:
import os
import warnings
import numpy as np
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim

import mlflow
import mlflow.pytorch
from mlflow.models.signature import infer_signature
from sklearn.metrics import f1_score, roc_auc_score, average_precision_score

warnings.filterwarnings("ignore", message=".*were assigned during export.*", category=UserWarning)

# --- Configuration ---
SEGMENT_LENGTH = 1250      # 5 seconds at 250 Hz
BATCH_SIZE = 64
LEARNING_RATE = 0.0001
NUM_EPOCHS = 50
PATIENCE = 10
POS_WEIGHT = 4.0           # ~80/20 negative-to-positive ratio
VT_LABELS = {"VT", "VF/VFL"}  # Labels considered "positive" (malignant arrhythmia)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
)
print(f"Device: {device}")
print(f"Segment length: {SEGMENT_LENGTH} samples ({SEGMENT_LENGTH/250:.1f}s at 250 Hz)")
print(f"Batch size: {BATCH_SIZE}, LR: {LEARNING_RATE}, Epochs: {NUM_EPOCHS}, Patience: {PATIENCE}")

In [0]:
# =============================================================================
# Dataset: Load CUDB segments from Delta table (with augmentation)
# =============================================================================
from torch.utils.data import Dataset, DataLoader

class CUDBDataset(Dataset):
    """
    PyTorch Dataset for CUDB ECG segments.
    Loads data from the Delta table and creates fixed-length segments
    with binary labels (1 = VT/VF, 0 = normal).
    
    When augment=True, applies random ECG augmentations at training time:
      - Gaussian noise injection
      - Amplitude scaling (gain jitter)
      - Baseline wander (low-frequency drift)
      - Random time shift (circular)
      - Signal dropout (electrode artifact simulation)
    
    Also generates synthetic constant-voltage segments (lead disconnect)
    stored in post-global-normalization space so the model learns to reject
    flat-line artifacts regardless of the inference normalization pipeline.
    """

    def __init__(self, df_pandas, segment_length=1250, vt_labels=None, augment=False,
                 disconnect_ratio=0.15):
        """
        Args:
            df_pandas: Pandas DataFrame with columns [record_id, sample_index, voltage_mv, label]
            segment_length: Number of samples per segment (default 1250 = 5s at 250Hz)
            vt_labels: Set of label strings considered positive class
            augment: If True, apply random augmentations during __getitem__
            disconnect_ratio: Fraction of dataset size to add as constant-voltage segments
        """
        self.segment_length = segment_length
        self.vt_labels = vt_labels or {"VT", "VF/VFL"}
        self.augment = augment

        # Group by record and build contiguous segments
        self.segments = []
        self.labels = []
        self.prenormalized = []  # True = skip per-segment normalization in __getitem__
        skipped_nan = 0

        for record_id, group in df_pandas.groupby("record_id"):
            group = group.sort_values("sample_index").reset_index(drop=True)
            signal = group["voltage_mv"].values.astype(np.float32)
            label_arr = group["label"].values

            # Slide non-overlapping windows
            n_segments = len(signal) // segment_length
            for i in range(n_segments):
                start = i * segment_length
                end = start + segment_length
                seg = signal[start:end]

                # Skip segments containing NaN or Inf
                if np.isnan(seg).any() or np.isinf(seg).any():
                    skipped_nan += 1
                    continue

                seg_labels = label_arr[start:end]

                # Binary label: positive if >50% of segment is VT/VF
                vt_fraction = np.isin(seg_labels, list(self.vt_labels)).mean()
                label = 1.0 if vt_fraction > 0.5 else 0.0

                self.segments.append(seg)
                self.labels.append(label)
                self.prenormalized.append(False)

        n_real = len(self.labels)

        # Add synthetic constant-voltage segments (lead disconnect simulation)
        # These are stored in POST-GLOBAL-NORMALIZATION space:
        # i.e., the values the model will see after global z-score at inference.
        # This way the model learns to reject flat lines at any amplitude level
        # without requiring per-window normalization at the API/inference layer.
        n_disconnect = 0
        if augment and disconnect_ratio > 0:
            n_disconnect = int(n_real * disconnect_ratio)
            for _ in range(n_disconnect):
                # Simulate what constant-voltage segments look like AFTER global
                # z-score normalization: (constant - global_mean) / global_std
                # Real examples from encounter data:
                #   constant 500 µV, global mean~11, std~106 → ~4.6
                #   constant 0 µV   → ~-0.1
                #   constant 2048   → ~19.2
                #   constant -500   → ~-4.8
                # We sample from a wide range to generalize across recordings.

                pattern = np.random.choice(
                    ["high_constant", "low_constant", "near_zero", "extreme", "noisy_flat"],
                    p=[0.3, 0.2, 0.2, 0.1, 0.2]
                )

                if pattern == "high_constant":
                    # Sentinel values above the mean (common: 500, 1000, 2048 raw)
                    const_val = np.random.uniform(2.0, 8.0)
                    seg = np.full(segment_length, const_val, dtype=np.float32)
                elif pattern == "low_constant":
                    # Negative constants (lead floating below mean)
                    const_val = np.random.uniform(-6.0, -1.0)
                    seg = np.full(segment_length, const_val, dtype=np.float32)
                elif pattern == "near_zero":
                    # Near the global mean (small offset after normalization)
                    const_val = np.random.uniform(-0.5, 0.5)
                    seg = np.full(segment_length, const_val, dtype=np.float32)
                elif pattern == "extreme":
                    # Very high sentinel values (e.g., 2048, 4096 raw)
                    const_val = np.random.uniform(10.0, 25.0)
                    seg = np.full(segment_length, const_val, dtype=np.float32)
                else:  # noisy_flat
                    # Near-constant with tiny measurement noise (realistic disconnect)
                    const_val = np.random.uniform(-5.0, 10.0)
                    seg = np.full(segment_length, const_val, dtype=np.float32)
                    seg += np.random.randn(segment_length).astype(np.float32) * 0.02

                self.segments.append(seg)
                self.labels.append(0.0)  # Always negative — not a real arrhythmia
                self.prenormalized.append(True)  # Already in normalized space

        self.segments = np.array(self.segments)
        self.labels = np.array(self.labels, dtype=np.float32)
        self.prenormalized = np.array(self.prenormalized, dtype=bool)

        n_pos = int(self.labels.sum())
        n_neg = len(self.labels) - n_pos
        print(f"  Segments: {len(self.labels)} (pos={n_pos}, neg={n_neg}, ratio={n_pos/len(self.labels)*100:.1f}%)")
        if skipped_nan > 0:
            print(f"  Skipped {skipped_nan} segments containing NaN/Inf")
        if augment:
            print(f"  Augmentation: ENABLED (noise, scale, wander, shift, dropout)")
        if n_disconnect > 0:
            print(f"  Lead disconnect segments: {n_disconnect} (global-norm-aware, labeled negative)")

    def __len__(self):
        return len(self.segments)

    def _augment(self, seg):
        """Apply random ECG augmentations to a normalized segment."""
        # 1. Gaussian noise (p=0.8, SNR ~20-40 dB)
        if np.random.random() < 0.8:
            noise_std = np.random.uniform(0.01, 0.15)
            seg = seg + np.random.randn(len(seg)).astype(np.float32) * noise_std

        # 2. Amplitude scaling / gain jitter (p=0.7, ±20%)
        if np.random.random() < 0.7:
            scale = np.random.uniform(0.8, 1.2)
            seg = seg * scale

        # 3. Baseline wander — low-frequency sinusoidal drift (p=0.5)
        if np.random.random() < 0.5:
            freq = np.random.uniform(0.1, 0.5)  # 0.1-0.5 Hz
            amplitude = np.random.uniform(0.05, 0.3)
            phase = np.random.uniform(0, 2 * np.pi)
            t = np.linspace(0, len(seg) / 250.0, len(seg), dtype=np.float32)
            wander = amplitude * np.sin(2 * np.pi * freq * t + phase)
            seg = seg + wander

        # 4. Random time shift — circular (p=0.5)
        if np.random.random() < 0.5:
            shift = np.random.randint(-len(seg) // 4, len(seg) // 4)
            seg = np.roll(seg, shift)

        # 5. Signal dropout — zero out a short segment (p=0.3, simulates artifact)
        if np.random.random() < 0.3:
            dropout_len = np.random.randint(10, 50)  # 40-200ms at 250Hz
            start_idx = np.random.randint(0, len(seg) - dropout_len)
            seg[start_idx:start_idx + dropout_len] = 0.0

        return seg

    def __getitem__(self, idx):
        seg = self.segments[idx].copy()  # copy to avoid mutating stored data

        if self.prenormalized[idx]:
            # Synthetic segment already in post-global-norm space — skip normalization
            # Only clamp to prevent overflow
            seg = np.clip(seg, -10.0, 10.0)
        else:
            # Normalize segment (z-score) with safe std
            seg_mean = seg.mean()
            seg_std = seg.std()
            if seg_std < 1e-6:
                seg = seg - seg_mean
            else:
                seg = (seg - seg_mean) / seg_std

            # Apply augmentation (after normalization, on standardized signal)
            if self.augment:
                seg = self._augment(seg)

            # Clamp to prevent extreme outliers from causing overflow
            seg = np.clip(seg, -10.0, 10.0)

        # Shape: (1, segment_length) for single-channel input
        x = torch.tensor(seg, dtype=torch.float32).unsqueeze(0)
        y = torch.tensor([self.labels[idx]], dtype=torch.float32)
        return x, y

In [0]:
# =============================================================================
# Load data from Delta table and create train/val splits
# =============================================================================

print("Loading CUDB data from Delta table...")
cudb_df = spark.read.table("biomedicalinformatics_analytics.dev_lab_deo.cudb") \
    .select("record_id", "sample_index", "voltage_mv", "label") \
    .toPandas()

print(f"Total samples: {len(cudb_df):,}")
print(f"Records: {cudb_df['record_id'].nunique()}")
print(f"Labels: {cudb_df['label'].value_counts().to_dict()}")

# Train/val split by record (80/20)
records = cudb_df["record_id"].unique()
np.random.seed(42)
np.random.shuffle(records)
split_idx = int(len(records) * 0.8)
train_records = set(records[:split_idx])
val_records = set(records[split_idx:])

train_pdf = cudb_df[cudb_df["record_id"].isin(train_records)]
val_pdf = cudb_df[cudb_df["record_id"].isin(val_records)]

print(f"\nTrain records: {len(train_records)}, Val records: {len(val_records)}")

print("\nBuilding training dataset...")
train_dataset = CUDBDataset(train_pdf, segment_length=SEGMENT_LENGTH, vt_labels=VT_LABELS, augment=True)
print("Building validation dataset...")
val_dataset = CUDBDataset(val_pdf, segment_length=SEGMENT_LENGTH, vt_labels=VT_LABELS, augment=False)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)

print(f"\nTrain batches: {len(train_loader)}, Val batches: {len(val_loader)}")

In [0]:
# =============================================================================
# Model Architecture: WFDBResNetLSTM (1-channel, binary classification)
# =============================================================================
import torch.nn.functional as F


class SEBlock(nn.Module):
    def __init__(self, channel, reduction=16):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Sequential(
            nn.Linear(channel, channel // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(channel // reduction, channel, bias=False),
            nn.Sigmoid(),
        )

    def forward(self, x):
        b, c, _ = x.size()
        y = self.avg_pool(x).view(b, c)
        y = self.fc(y).view(b, c, 1)
        return x * y.expand_as(x)


class BasicBlock1d(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        self.conv1 = nn.Conv1d(in_channels, out_channels, kernel_size=7, stride=stride, padding=3, bias=False)
        self.bn1 = nn.BatchNorm1d(out_channels)
        self.conv2 = nn.Conv1d(out_channels, out_channels, kernel_size=7, stride=1, padding=3, bias=False)
        self.bn2 = nn.BatchNorm1d(out_channels)
        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv1d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm1d(out_channels),
            )
        self.se = SEBlock(out_channels)

    def forward(self, x):
        identity = self.shortcut(x)
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out = self.se(out)
        out += identity
        return F.relu(out)


class WFDBResNetLSTM(nn.Module):
    """ResNet-LSTM for single-lead ECG binary classification (VT detection)."""

    def __init__(self, in_channels=1, num_classes=1):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv1d(in_channels, 32, kernel_size=15, stride=3, padding=7, bias=False),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=3, stride=2, padding=1),
        )
        self.layer1 = self._make_layer(32, 64, 2, stride=1)
        self.layer2 = self._make_layer(64, 128, 2, stride=2)
        self.layer3 = self._make_layer(128, 256, 2, stride=2)
        self.lstm = nn.LSTM(input_size=256, hidden_size=128, num_layers=2,
                            batch_first=True, bidirectional=True)
        self.attention = nn.Linear(256, 1)
        self.dropout = nn.Dropout(p=0.5)
        self.fc = nn.Linear(256, num_classes)

    def _make_layer(self, in_ch, out_ch, num_blocks, stride):
        layers = [BasicBlock1d(in_ch, out_ch, stride)]
        for _ in range(1, num_blocks):
            layers.append(BasicBlock1d(out_ch, out_ch, stride=1))
        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.stem(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = x.permute(0, 2, 1)  # (batch, seq_len, 256)
        lstm_out, _ = self.lstm(x)
        attn_weights = torch.softmax(self.attention(lstm_out), dim=1)
        context = torch.sum(attn_weights * lstm_out, dim=1)
        x = self.dropout(context)
        return self.fc(x)


model = WFDBResNetLSTM(in_channels=1, num_classes=1).to(device)
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
print(model)

In [0]:
# =============================================================================
# Optimizer, loss, and scheduler
# =============================================================================

pos_weight = torch.tensor([POS_WEIGHT]).to(device)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS, eta_min=1e-6)

print(f"Loss: BCEWithLogitsLoss (pos_weight={POS_WEIGHT})")
print(f"Optimizer: Adam (lr={LEARNING_RATE})")
print(f"Scheduler: CosineAnnealing (T_max={NUM_EPOCHS}, eta_min=1e-6)")
print(f"Early stopping patience: {PATIENCE}")

In [0]:
# =============================================================================
# Training loop with validation and early stopping
# =============================================================================

best_val_loss = float("inf")
best_model_state = None
epochs_no_improve = 0

mlflow.set_experiment("/Shared/cudb_classifier")
mlflow.end_run()  # End any stale active run from a previous execution

with mlflow.start_run():
    mlflow.log_params({
        "learning_rate": LEARNING_RATE,
        "num_epochs": NUM_EPOCHS,
        "batch_size": BATCH_SIZE,
        "segment_length": SEGMENT_LENGTH,
        "pos_weight": POS_WEIGHT,
        "patience": PATIENCE,
        "model_type": "WFDBResNetLSTM_1ch",
    })

    print(f"Starting training for {NUM_EPOCHS} epochs...\n")

    for epoch in range(NUM_EPOCHS):
        # --- Training ---
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0

        train_pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS} [Train]")
        for inputs, targets in train_pbar:
            inputs, targets = inputs.to(device), targets.to(device)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            running_loss += loss.item()
            predicted = (torch.sigmoid(outputs) > 0.5).float()
            correct += (predicted == targets).sum().item()
            total += targets.size(0)
            train_pbar.set_postfix({"loss": f"{loss.item():.4f}"})

        avg_loss = running_loss / len(train_loader)
        train_acc = 100.0 * correct / total

        # --- Validation ---
        model.eval()
        val_loss = 0.0
        val_correct = 0
        val_total = 0
        all_val_targets = []
        all_val_probs = []

        with torch.no_grad():
            for val_inputs, val_targets in val_loader:
                val_inputs, val_targets = val_inputs.to(device), val_targets.to(device)
                outputs = model(val_inputs)
                loss = criterion(outputs, val_targets)
                val_loss += loss.item()

                probs = torch.sigmoid(outputs)
                predicted = (probs > 0.5).float()
                val_correct += (predicted == val_targets).sum().item()
                val_total += val_targets.size(0)

                all_val_targets.extend(val_targets.cpu().numpy().flatten())
                all_val_probs.extend(probs.cpu().numpy().flatten())

        avg_val_loss = val_loss / len(val_loader)
        val_acc = 100.0 * val_correct / val_total

        val_targets_np = np.array(all_val_targets)
        val_preds_np = (np.array(all_val_probs) > 0.5).astype(float)
        val_f1 = f1_score(val_targets_np, val_preds_np, zero_division=0)

        scheduler.step()

        # --- Logging ---
        mlflow.log_metrics({
            "train_loss": avg_loss,
            "train_acc": train_acc,
            "val_loss": avg_val_loss,
            "val_acc": val_acc,
            "val_f1": val_f1,
            "lr": scheduler.get_last_lr()[0],
        }, step=epoch)

        print(f"Epoch [{epoch+1}/{NUM_EPOCHS}] "
              f"Train Loss: {avg_loss:.4f}, Acc: {train_acc:.1f}% | "
              f"Val Loss: {avg_val_loss:.4f}, Acc: {val_acc:.1f}%, F1: {val_f1:.4f}")

        # --- Early stopping ---
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            epochs_no_improve = 0
            best_model_state = model.state_dict().copy()
            print(f"  --> Best model saved (val_loss={best_val_loss:.4f})")
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= PATIENCE:
                print(f"\nEarly stopping at epoch {epoch+1} (no improvement for {PATIENCE} epochs)")
                break

    print("\nTraining complete!")

In [0]:
# =============================================================================
# Threshold calibration and final evaluation
# =============================================================================

# Load best model
if best_model_state is None:
    print("WARNING: No best model state saved (training may have produced NaN losses).")
    print("Using current model weights for evaluation.")
    best_model_state = model.state_dict()
else:
    model.load_state_dict(best_model_state)
model.eval()

# Collect predictions on validation set
all_targets = []
all_probs = []

with torch.no_grad():
    for val_inputs, val_targets in val_loader:
        outputs = model(val_inputs.to(device))
        probs = torch.sigmoid(outputs).cpu().numpy()
        all_probs.extend(probs.flatten())
        all_targets.extend(val_targets.numpy().flatten())

all_probs = np.array(all_probs)
all_targets = np.array(all_targets)

# Sweep thresholds to find optimal F1
best_threshold = 0.5
best_f1 = 0.0
for thresh in np.arange(0.1, 0.9, 0.05):
    preds = (all_probs > thresh).astype(float)
    score = f1_score(all_targets, preds, zero_division=0)
    if score > best_f1:
        best_f1 = score
        best_threshold = thresh

print(f"OPTIMAL THRESHOLD: {best_threshold:.2f}")
print(f"Best F1 at threshold: {best_f1:.4f}")

# Compute AUC metrics
try:
    roc_auc = roc_auc_score(all_targets, all_probs)
    pr_auc = average_precision_score(all_targets, all_probs)
    print(f"ROC-AUC: {roc_auc:.4f}")
    print(f"PR-AUC:  {pr_auc:.4f}")
    mlflow.log_metric("final_roc_auc", float(roc_auc))
    mlflow.log_metric("final_pr_auc", float(pr_auc))
except Exception as e:
    print(f"Could not calculate AUC: {e}")

mlflow.log_metric("optimal_threshold", best_threshold)
mlflow.log_metric("best_f1", best_f1)

In [0]:
# =============================================================================
# Save model checkpoint and register with MLflow
# =============================================================================

# Save local checkpoint
checkpoint_path = "/Workspace/DeoLab/Malignant Arrhythmia Analysis/best_cudb_resnet_lstm.pth"
torch.save(best_model_state, checkpoint_path)
print(f"Checkpoint saved: {checkpoint_path}")

# Log model to MLflow with signature
sample_input = next(iter(val_loader))[0][0:2].numpy()
model_cpu = model.to("cpu")
sample_output = model_cpu(torch.FloatTensor(sample_input)).detach().numpy()
signature = infer_signature(sample_input, sample_output)

mlflow.pytorch.log_model(
    pytorch_model=model_cpu,
    name="cudb_crnn_model",
    signature=signature,
    input_example=sample_input,
    registered_model_name="biomedicalinformatics_analytics.prepared.CUDB_Malignant_Classifier",
)

print(f"\nModel registered as 'biomedicalinformatics_analytics.prepared.CUDB_Malignant_Classifier'")
print(f"Optimal threshold: {best_threshold:.2f}")
print("Done!")